# Delta Lake 4.0 — Combined Notebook

This notebook merges all topics from the original Databricks export (`DeltaLake4.0.dbc`) into a single sequential notebook.

**Note:** cells reference Databricks-managed tables/volumes (e.g. `deltalakeansh.default.clonetbl`, `/Volumes/deltalakeansh/...`). Outside Databricks you'll need a Delta-enabled Spark session and your own table/path names.


## Table of Contents
1. [1 — Create Delta Table](#1---create-delta-table)
2. [2 — Delta Log](#2---delta-log)
3. [3 — Schema Enforcement](#3---schema-enforcement)
4. [4 — DML (Update / Delete / Merge)](#4---dml-update--delete--merge)
5. [5 — Delta Log Level Schema Changes](#5---delta-log-level-schema-changes)
6. [6 — Table Utility Commands](#6---table-utility-commands)
7. [7 — Change Data Feed (CDF)](#7---change-data-feed-cdf)
8. [8 — Uniform](#8---uniform)
9. [9 — Optimization](#9---optimization)


---
## 1 — Create Delta Table


In [ ]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

# **Create Delta Table**

### **Using SQL API**

In [ ]:
# %sql
CREATE TABLE deltalakeansh.default.first_table
(
  id INT NOT NULL,
  salary INT UNIQUE
)

### **Using Delta API**

In [ ]:
from delta.tables import DeltaTable, IdentityGenerator

Error: [0;31m---------------------------------------------------------------------------[0m
[0;31mImportError[0m                               Traceback (most recent call last)
File [0;32m<command-5420445660553522>, line 1[0m
[0;32m----> 1[0m [38;5;28;01mfrom[39;00m [38;5;21;01mdelta[39;00m[38;5;21;01m.[39;00m[38;5;21;01mtables[39;00m [38;5;28;01mimport[39;00m DeltaTable, IdentityGenerator

[0;31mImportError[0m: cannot import name 'IdentityGenerator' from 'delta.tables' (/databricks/python/lib/python3.11/site-packages/delta/tables.py)

In [ ]:
DeltaTable.createIfNotExists(spark) \
  .tableName("deltalakeansh.default.firstdeltaapi") \
  .addColumn("id", "INT") \
  .addColumn("salary", "INT") \
  .execute()

{"text/plain": "<delta.connect.tables.DeltaTable at 0xfff8041e83d0>"}

In [ ]:
# %sql
DROP TABLE deltalakeansh.default.firstdeltaapi

# **GENERATED COLUMNS**

### **Identity Columns**

In [ ]:
DeltaTable.create(spark)\
  .tableName("deltalakeansh.default.firstdeltaapi")\
  .addColumn("id_col", dataType=LongType(), generatedAlwaysAs=IdentityGenerator())\
  .addColumn("salary", dataType = IntegerType())\
  .addColumn("name", dataType = StringType())\
  .execute()

Error: [0;31m---------------------------------------------------------------------------[0m
[0;31mNameError[0m                                 Traceback (most recent call last)
File [0;32m<command-5420445660553524>, line 3[0m
[1;32m      1[0m DeltaTable[38;5;241m.[39mcreate(spark)\
[1;32m      2[0m   [38;5;241m.[39mtableName([38;5;124m"[39m[38;5;124mdeltalakeansh.default.firstdeltaapi[39m[38;5;124m"[39m)\
[0;32m----> 3[0m   [38;5;241m.[39maddColumn([38;5;124m"[39m[38;5;124mid_col[39m[38;5;124m"[39m, dataType[38;5;241m=[39mLongType(), generatedAlwaysAs[38;5;241m=[39mIdentityGenerator())\
[1;32m      4[0m   [38;5;241m.[39maddColumn([38;5;124m"[39m[38;5;124msalary[39m[38;5;124m"[39m, dataType [38;5;241m=[39m IntegerType())\
[1;32m      5[0m   [38;5;241m.[39maddColumn([38;5;124m"[39m[38;5;124mname[39m[38;5;124m"[39m, dataType [38;5;241m=[39m StringType())\
[1;32m      6[0m   [38;5;241m.[39mexecute()

[0;31mNameError[0m: name '

### **Computed Columns**

In [ ]:
DeltaTable.create(spark)\
  .tableName("deltalakeansh.default.firstdeltaapi")\
  .addColumn("salaryAfterTax", dataType=LongType(), generatedAlwaysAs = "CAST((salary * 0.7) AS BIGINT)")\
  .addColumn("salary", dataType = IntegerType())\
  .addColumn("name", dataType = StringType())\
  .execute()

{"text/plain": "<delta.connect.tables.DeltaTable at 0xfff8036981d0>"}

In [ ]:
# %sql
INSERT INTO deltalakeansh.default.firstdeltaapi(salary,name)
VALUES 
(100,'aa'),
(200,'bb')

num_affected_rows | num_inserted_rows
--- | ---
2 | 2

In [ ]:
# %sql
SELECT * FROM deltalakeansh.default.firstdeltaapi

salaryAfterTax | salary | name
--- | --- | ---
70 | 100 | aa
140 | 200 | bb

---
## 2 — Delta Log


# **DELTA LOG**

In [ ]:
data = [(1,100,'aa'),(2,200,'bb'),(3,300,'cc'),(4,400,'dd')]

df = spark.createDataFrame(data, ['id','salary','name'])

display(df)

id | salary | name
--- | --- | ---
1 | 100 | aa
2 | 200 | bb
3 | 300 | cc
4 | 400 | dd

In [ ]:
df.write.format("delta")\
        .mode("append")\
        .save("/Volumes/deltalakeansh/default/deltavol/demosink/")

In [ ]:
df = spark.read.format("json")\
            .load("/Volumes/deltalakeansh/default/deltavol/demosink/_delta_log/00000000000000000001.json")
display(df)

add | commitInfo
--- | ---
None | ['0705-000106-t0srhzg8-v2n', 'Databricks-Runtime/16.4.x-aarch64-photon-scala2.12', True, 'WriteSerializable', 'WRITE', ['1', '1066', '4'], ['Append', '[]', False], 0, ['true', 'false'], 1751676225144, '823ff9e1-fbdf-4325-a47a-fbf7c79f48e7', '4512597665796397', 'anshlambaaz@gmail.com']
[True, 1751676225000, 'part-00000-5b3ff767-281f-4e85-a249-065a808de0ce.c000.snappy.parquet', 1066, '{"numRecords":4,"minValues":{"id":1,"salary":100,"name":"aa"},"maxValues":{"id":4,"salary":400,"name":"dd"},"nullCount":{"id":0,"salary":0,"name":0},"tightBounds":true}', ['1751676225000000', '1751676225000000', '1751676225000000', '268435456']] | None

---
## 3 — Schema Enforcement


# **SCHEMA ENFORCEMENT**

In [ ]:
data = [(1,100,'aa',1),(2,200,'bb',1),(3,300,'cc',1),(4,400,'dd',1)]

df = spark.createDataFrame(data, ['id','salary','name','sus_col'])

display(df)

id | salary | name | sus_col
--- | --- | --- | ---
1 | 100 | aa | 1
2 | 200 | bb | 1
3 | 300 | cc | 1
4 | 400 | dd | 1

In [ ]:
df.write.format("delta")\
        .mode("append")\
        .save("/Volumes/deltalakeansh/default/deltavol/demosink/")

Error: [0;31m---------------------------------------------------------------------------[0m
[0;31mAnalysisException[0m                         Traceback (most recent call last)
File [0;32m<command-7041518322875396>, line 3[0m
[1;32m      1[0m df[38;5;241m.[39mwrite[38;5;241m.[39mformat([38;5;124m"[39m[38;5;124mdelta[39m[38;5;124m"[39m)\
[1;32m      2[0m         [38;5;241m.[39mmode([38;5;124m"[39m[38;5;124mappend[39m[38;5;124m"[39m)\
[0;32m----> 3[0m         [38;5;241m.[39msave([38;5;124m"[39m[38;5;124m/Volumes/deltalakeansh/default/deltavol/demosink/[39m[38;5;124m"[39m)

File [0;32m/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/readwriter.py:679[0m, in [0;36mDataFrameWriter.save[0;34m(self, path, format, mode, partitionBy, **options)[0m
[1;32m    677[0m     [38;5;28mself[39m[38;5;241m.[39mformat([38;5;28mformat[39m)
[1;32m    678[0m [38;5;28mself[39m[38;5;241m.[39m_write[38;5;241m.[39mpath [38;5;241m=[39m

# **SCHEMA EVOLUTION**

In [ ]:
df.write.format("delta")\
        .mode("append")\
        .option("mergeSchema",True)\
        .save("/Volumes/deltalakeansh/default/deltavol/demosink/")

# READ DELTA DATA

### **TABLE**

In [ ]:
# %sql
SELECT * FROM deltalakeansh.default.firstdeltaapi

salaryAfterTax | salary | name
--- | --- | ---
70 | 100 | aa
140 | 200 | bb

### **DATA LAKE**

In [ ]:
# %sql
SELECT * FROM delta.`/Volumes/deltalakeansh/default/deltavol/demosink/`

cust_id | income | name | tip
--- | --- | --- | ---
1 | 100 | aa | 1
2 | 200 | bb | 1
3 | 300 | cc | 1
4 | 400 | dd | 1

# **SCHEMA OVERWRITE**

In [ ]:
data = [(1,100,'aa',1),(2,200,'bb',1),(3,300,'cc',1),(4,400,'dd',1)]

df = spark.createDataFrame(data, ['cust_id','income','name','tip'])

display(df)

cust_id | income | name | tip
--- | --- | --- | ---
1 | 100 | aa | 1
2 | 200 | bb | 1
3 | 300 | cc | 1
4 | 400 | dd | 1

In [ ]:
df.write.format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema",True)\
        .save("/Volumes/deltalakeansh/default/deltavol/demosink/")

---
## 4 — DML (Update / Delete / Merge)


# **DML**

In [ ]:
data = [(5,100,'aa',1),(6,200,'bb',1),(7,300,'cc',1),(8,400,'dd',1)]

df = spark.createDataFrame(data, ['cust_id','income','name','tip'])

df.write.format("delta")\
        .mode("append")\
        .save("/Volumes/deltalakeansh/default/deltavol/dmlsink/")

In [ ]:
# %sql
SELECT * FROM delta.`/Volumes/deltalakeansh/default/deltavol/dmlsink/`

cust_id | income | name | tip
--- | --- | --- | ---
5 | 100 | aa | 1
6 | 200 | bb | 1
7 | 300 | cc | 1
8 | 400 | dd | 1
1 | 100 | aa | 1
2 | 200 | bb | 1
3 | 300 | cc | 1
4 | 400 | dd | 1

### **update**

In [ ]:
# %sql
UPDATE delta.`/Volumes/deltalakeansh/default/deltavol/dmlsink/`
SET income = 1000 WHERE cust_id = 5

num_affected_rows
---
1

In [ ]:
# %sql
SELECT * FROM delta.`/Volumes/deltalakeansh/default/deltavol/dmlsink/`

cust_id | income | name | tip
--- | --- | --- | ---
1 | 100 | aa | 1
2 | 200 | bb | 1
3 | 300 | cc | 1
4 | 400 | dd | 1
5 | 1000 | aa | 1
6 | 200 | bb | 1
7 | 300 | cc | 1
8 | 400 | dd | 1

# **UPSERT**

In [ ]:
data = [(1,100,'xyz',1),(9,200,'bb',1),(10,300,'cc',1)]

df = spark.createDataFrame(data, ['cust_id','income','name','tip'])

display(df)


cust_id | income | name | tip
--- | --- | --- | ---
1 | 100 | xyz | 1
9 | 200 | bb | 1
10 | 300 | cc | 1

In [ ]:
from delta.tables import DeltaTable

In [ ]:
dlt_obj = DeltaTable.forPath(spark,"/Volumes/deltalakeansh/default/deltavol/dmlsink/")

dlt_obj.alias("trg").merge(df.alias("src"),"trg.cust_id = src.cust_id")\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()

{"text/plain": "DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]"}

In [ ]:
# %sql
SELECT * FROM delta.`/Volumes/deltalakeansh/default/deltavol/dmlsink/`

cust_id | income | name | tip
--- | --- | --- | ---
2 | 200 | bb | 1
3 | 300 | cc | 1
4 | 400 | dd | 1
5 | 1000 | aa | 1
6 | 200 | bb | 1
7 | 300 | cc | 1
8 | 400 | dd | 1
1 | 100 | xyz | 1
9 | 200 | bb | 1
10 | 300 | cc | 1

---
## 5 — Delta Log Level Schema Changes


# **DELTA LOG LEVEL SCHEMA CHANGES**

In [ ]:
data = [(1,100,'xyz',1),(9,200,'bb',1),(10,300,'cc',1)]

df = spark.createDataFrame(data, ['cust_id','income','name','tip'])

display(df)


cust_id | income | name | tip
--- | --- | --- | ---
1 | 100 | xyz | 1
9 | 200 | bb | 1
10 | 300 | cc | 1

In [ ]:
df.write.format("delta")\
        .mode("append")\
        .save("/Volumes/deltalakeansh/default/deltavol/schemalevel/")

In [ ]:
# %sql
-- Enable column mapping on the Delta table
ALTER TABLE delta.`/Volumes/deltalakeansh/default/deltavol/schemalevel/`
SET TBLPROPERTIES (
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5',
  'delta.columnMapping.mode' = 'name'
);

-- Rename the column
ALTER TABLE delta.`/Volumes/deltalakeansh/default/deltavol/schemalevel/`
RENAME COLUMN name TO customer_name;

In [ ]:
# %sql
SELECT * FROM delta.`/Volumes/deltalakeansh/default/deltavol/schemalevel/`

cust_id | income | customer_name | tip
--- | --- | --- | ---
1 | 100 | xyz | 1
9 | 200 | bb | 1
10 | 300 | cc | 1

In [ ]:
df = spark.read.format("json")\
            .load("/Volumes/deltalakeansh/default/deltavol/schemalevel/_delta_log/00000000000000000002.json")
df.display()

commitInfo | metaData
--- | ---
['0705-124850-z6q6t96p-v2n', 'Databricks-Runtime/16.4.x-aarch64-photon-scala2.12', True, 'WriteSerializable', 'RENAME COLUMN', ['customer_name', 'name'], 1, 1751720750285, '92ef2aeb-542f-4d5f-a5a6-424613c61ef8', '4512597665796397', 'anshlambaaz@gmail.com'] | None
None | [['4', 'name', 'true'], 1751720247220, ['parquet'], '3fc253b2-b52f-4e55-8191-7f8b406a7d9f', [], '{"type":"struct","fields":[{"name":"cust_id","type":"long","nullable":true,"metadata":{"delta.columnMapping.id":1,"delta.columnMapping.physicalName":"cust_id"}},{"name":"income","type":"long","nullable":true,"metadata":{"delta.columnMapping.id":2,"delta.columnMapping.physicalName":"income"}},{"name":"customer_name","type":"string","nullable":true,"metadata":{"delta.columnMapping.id":3,"delta.columnMapping.physicalName":"name"}},{"name":"tip","type":"long","nullable":true,"metadata":{"delta.columnMapping.id":4,"delta.columnMapping.physicalName":"tip"}}]}']

---
## 6 — Table Utility Commands


# **TABLE UTILITY COMMANDS**

In [ ]:
# %sql
DESCRIBE DETAIL deltalakeansh.default.first_table

format | id | name | description | location | createdAt | lastModified | partitionColumns | clusteringColumns | numFiles | sizeInBytes | properties | minReaderVersion | minWriterVersion | tableFeatures | statistics | clusterByAuto
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
delta | 25f7989d-0263-4857-82d1-3a2dcb6ffb22 | deltalakeansh.default.first_table | None |  | 2025-07-04T22:37:04.805Z | 2025-07-04T22:37:06.000Z | [] | [] | 0 | 0 | {'delta.enableDeletionVectors': 'true'} | 3 | 7 | ['appendOnly', 'deletionVectors', 'invariants'] | {'numRowsDeletedByDeletionVectors': 0, 'numDeletionVectors': 0} | False

In [ ]:
# %sql
DESCRIBE EXTENDED deltalakeansh.default.first_table

col_name | data_type | comment
--- | --- | ---
id | int | None
salary | int | None
 |  | 
# Detailed Table Information |  | 
Catalog | deltalakeansh | 
Database | default | 
Table | first_table | 
Created Time | Fri Jul 04 22:37:06 UTC 2025 | 
Last Access | UNKNOWN | 
Created By | Spark  | 
Type | MANAGED | 
Location |  | 
Provider | delta | 
Owner | anshlambaaz@gmail.com | 
Is_managed_location | true | 
Predictive Optimization | ENABLE (inherited from METASTORE metastore_aws_us_east_2) | 
Table Properties | [delta.enableDeletionVectors=true,delta.feature.appendOnly=supported,delta.feature.deletionVectors=supported,delta.feature.invariants=supported,delta.minReaderVersion=3,delta.minWriterVersion=7] | 

In [ ]:
# %sql
DESCRIBE HISTORY delta.`/Volumes/deltalakeansh/default/deltavol/dmlsink/`

version | timestamp | userId | userName | operation | operationParameters | job | notebook | clusterId | readVersion | isolationLevel | isBlindAppend | operationMetrics | userMetadata | engineInfo
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
5 | 2025-07-05T12:52:04.000Z | 4512597665796397 | anshlambaaz@gmail.com | OPTIMIZE | {'predicate': '[]', 'auto': 'true', 'clusterBy': '[]', 'zOrderBy': '[]', 'batchId': '0'} | None | None | 0705-124850-z6q6t96p-v2n | 4 | SnapshotIsolation | False | {'numRemovedFiles': '4', 'numRemovedBytes': '5046', 'p25FileSize': '1413', 'numDeletionVectorsRemoved': '1', 'minFileSize': '1413', 'numAddedFiles': '1', 'maxFileSize': '1413', 'p75FileSize': '1413', 'p50FileSize': '1413', 'numAddedBytes': '1413'} | None | Databricks-Runtime/16.4.x-aarch64-photon-scala2.12
4 | 2025-07-05T12:52:00.000Z | 4512597665796397 | anshlambaaz@gmail.com | MERGE | {'predicate': '["(cust_id#10533L = cust_id#10485L)"]', 'clusterBy': '[]', 'm

In [ ]:
# %sql
SELECT * FROM delta.`/Volumes/deltalakeansh/default/deltavol/dmlsink/`
TIMESTAMP AS OF '2025-07-05T01:12:36.000+00:00'

cust_id | income | name | tip
--- | --- | --- | ---
1 | 100 | aa | 1
2 | 200 | bb | 1
3 | 300 | cc | 1
4 | 400 | dd | 1
5 | 1000 | aa | 1
6 | 200 | bb | 1
7 | 300 | cc | 1
8 | 400 | dd | 1

In [ ]:
# %sql
RESTORE delta.`/Volumes/deltalakeansh/default/deltavol/dmlsink/` 
TO VERSION AS OF 3

table_size_after_restore | num_of_files_after_restore | num_removed_files | num_restored_files | removed_files_size | restored_files_size
--- | --- | --- | --- | --- | ---
1391 | 1 | 1 | 1 | 1413 | 1391

In [ ]:
# %sql
SHOW TBLPROPERTIES delta.`/Volumes/deltalakeansh/default/deltavol/dmlsink`

key | value
--- | ---
delta.enableDeletionVectors | true
delta.feature.appendOnly | supported
delta.feature.deletionVectors | supported
delta.feature.invariants | supported
delta.minReaderVersion | 3
delta.minWriterVersion | 7

In [ ]:
# %sql
VACUUM delta.`/Volumes/deltalakeansh/default/deltavol/dmlsink/` RETAIN 0 HOURS

In [ ]:
# %sql
CREATE TABLE deltalakeansh.default.clonetblshallow
SHALLOW CLONE deltalakeansh.default.first_table

source_table_size | source_num_of_files | num_of_synced_transactions | num_removed_files | num_copied_files | removed_files_size | copied_files_size
--- | --- | --- | --- | --- | --- | ---
0 | 0 | None | 0 | 0 | 0 | 0

In [ ]:
# %sql
SELECT * FROM deltalakeansh.default.clonetbl

cust_id | income | name | tip
--- | --- | --- | ---
1 | 100 | aa | 1
2 | 200 | bb | 1
3 | 300 | cc | 1
4 | 400 | dd | 1
5 | 1000 | aa | 1
6 | 200 | bb | 1
7 | 300 | cc | 1
8 | 400 | dd | 1

---
## 7 — Change Data Feed (CDF)


In [ ]:
# %sql
ALTER TABLE deltalakeansh.default.clonetbl
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [ ]:
# %sql
SELECT * FROM deltalakeansh.default.clonetbl

cust_id | income | name | tip
--- | --- | --- | ---
1 | 100 | aa | 1
2 | 200 | bb | 1
3 | 300 | cc | 1
4 | 400 | dd | 1
5 | 1000 | aa | 1
6 | 200 | bb | 1
7 | 300 | cc | 1
8 | 400 | dd | 1

In [ ]:
# %sql
INSERT INTO deltalakeansh.default.clonetbl
VALUES (10, 100, 'zz',1)

num_affected_rows | num_inserted_rows
--- | ---
1 | 1

In [ ]:
# %sql
UPDATE deltalakeansh.default.clonetbl
SET name = 'hi bro' where cust_id = 10

num_affected_rows
---
1

In [ ]:
# %sql
DELETE FROM deltalakeansh.default.clonetbl
WHERE cust_id = 3

num_affected_rows
---
1

In [ ]:
# %sql
DESCRIBE HISTORY deltalakeansh.default.clonetbl

version | timestamp | userId | userName | operation | operationParameters | job | notebook | clusterId | readVersion | isolationLevel | isBlindAppend | operationMetrics | userMetadata | engineInfo
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
5 | 2025-07-05T14:17:07.000Z | 4512597665796397 | anshlambaaz@gmail.com | OPTIMIZE | {'predicate': '[]', 'auto': 'true', 'clusterBy': '[]', 'zOrderBy': '[]', 'batchId': '0'} | None | None | 0705-124850-z6q6t96p-v2n | 4 | SnapshotIsolation | False | {'numRemovedFiles': '2', 'numRemovedBytes': '2800', 'p25FileSize': '1411', 'numDeletionVectorsRemoved': '1', 'minFileSize': '1411', 'numAddedFiles': '1', 'maxFileSize': '1411', 'p75FileSize': '1411', 'p50FileSize': '1411', 'numAddedBytes': '1411'} | None | Databricks-Runtime/16.4.x-aarch64-photon-scala2.12
4 | 2025-07-05T14:17:05.000Z | 4512597665796397 | anshlambaaz@gmail.com | DELETE | {'predicate': '["(cust_id#18491L = 3)"]'} | None | None | 0705-124850-z6q6t

In [ ]:
# %sql
SELECT * FROM table_changes('deltalakeansh.default.clonetbl',1,3)

cust_id | income | name | tip | _change_type | _commit_version | _commit_timestamp
--- | --- | --- | --- | --- | --- | ---
10 | 100 | zz | 1 | update_preimage | 3 | 2025-07-05T14:16:42.000Z
10 | 100 | hi bro | 1 | update_postimage | 3 | 2025-07-05T14:16:42.000Z
10 | 100 | zz | 1 | insert | 2 | 2025-07-05T14:16:05.000Z

---
## 8 — Uniform


# **UNIFORM**

In [ ]:
# %sql
CREATE TABLE deltalakeansh.default.unitbl
USING DELTA TBLPROPERTIES(
  'delta.enableIcebergCompatV2' = 'true',
  'delta.universalFormat.enabledFormats' = 'iceberg');

In [ ]:
# %sql
INSERT INTO deltalakeansh.default.unitbl

---
## 9 — Optimization


In [ ]:
df = spark.read.table("deltalakeansh.default.clonetbl")

In [ ]:
display(df)

cust_id | income | name | tip
--- | --- | --- | ---
1 | 100 | aa | 1
2 | 200 | bb | 1
4 | 400 | dd | 1
5 | 1000 | aa | 1
6 | 200 | bb | 1
7 | 300 | cc | 1
8 | 400 | dd | 1
10 | 100 | hi bro | 1

In [ ]:
df.write.format("delta")\
        .mode("append")\
        .save("/Volumes/deltalakeansh/default/deltavol/optimization/")

In [ ]:
# %sql
OPTIMIZE delta.`/Volumes/deltalakeansh/default/deltavol/optimization/`

path | metrics
--- | ---
dbfs:/Volumes/deltalakeansh/default/deltavol/optimization | [1, 2, [1443, 1443, 1443.0, 1, 1443], [1411, 1411, 1411.0, 2, 2822], 0, None, None, 0, 1, 2, 0, True, 0, 0, 1751730480487, 1751730481796, 8, 1, None, [0, 0], None, 4, 4, 279, 0, None]

In [ ]:
# %sql
DESCRIBE HISTORY delta.`/Volumes/deltalakeansh/default/deltavol/optimization/`

version | timestamp | userId | userName | operation | operationParameters | job | notebook | clusterId | readVersion | isolationLevel | isBlindAppend | operationMetrics | userMetadata | engineInfo
--- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---
4 | 2025-07-05T15:48:02.000Z | 4512597665796397 | anshlambaaz@gmail.com | OPTIMIZE | {'predicate': '[]', 'auto': 'false', 'clusterBy': '[]', 'zOrderBy': '[]', 'batchId': '0'} | None | None | 0705-124850-z6q6t96p-v2n | 3 | SnapshotIsolation | False | {'numRemovedFiles': '2', 'numRemovedBytes': '2822', 'p25FileSize': '1443', 'numDeletionVectorsRemoved': '0', 'minFileSize': '1443', 'numAddedFiles': '1', 'maxFileSize': '1443', 'p75FileSize': '1443', 'p50FileSize': '1443', 'numAddedBytes': '1443'} | None | Databricks-Runtime/16.4.x-aarch64-photon-scala2.12
3 | 2025-07-05T15:47:46.000Z | 4512597665796397 | anshlambaaz@gmail.com | WRITE | {'mode': 'Append', 'statsOnLoad': 'false', 'partitionBy': '[]'} | None | No

In [ ]:
# %sql
SELECT * FROM delta.`/Volumes/deltalakeansh/default/deltavol/optimization/`
WHERE cust_id = 1

cust_id | income | name | tip
--- | --- | --- | ---
1 | 100 | aa | 1
1 | 100 | aa | 1

In [ ]:
# %sql
OPTIMIZE delta.`/Volumes/deltalakeansh/default/deltavol/optimization/` ZORDER BY (cust_id)

path | metrics
--- | ---
dbfs:/Volumes/deltalakeansh/default/deltavol/optimization | [0, 0, [None, None, 0.0, 0, 0], [None, None, 0.0, 0, 0], 0, ['minCubeSize(107374182400)', [0, 0], [1, 1443], 0, [0, 0], 0, None], None, 0, 0, 1, 1, False, 0, 0, 1751730887487, 1751730887882, 8, 0, None, [0, 0], None, 4, 4, 0, 0, None]

In [ ]:
# %sql
ALTER TABLE deltalakeansh.default.clonetbl
CLUSTER BY AUTO